# Capstone Project 1: Manufacturing Equipment Output Prediction (Linear Regression)

**Goal:** Predict `Parts_Per_Hour` (machine output) from operating parameters using Linear Regression.

This notebook is a cleaned-up, single-pipeline rebuild of the original project notebook. It fixes these issues found in the original:
- Categorical columns (`Shift`, `Machine_Type`, `Material_Grade`, `Day_of_Week`) were being **dropped instead of encoded**.
- Two different models existed in parallel (one numeric-only, one with encoding) — evaluation and coefficients didn't match.
- `Timestamp` was never handled in the main flow.
- Two duplicate Streamlit apps.
- Imports (`Ridge`, `Lasso`, `cross_val_score`) were unused.

**Note on evaluation metrics:** the project brief lists Recall/Precision/F1 as "primary metrics" — those are classification metrics and don't apply here since `Parts_Per_Hour` is continuous. This notebook uses the correct regression metrics: **R², RMSE, MSE, MAE**.

Run the cells top to bottom in Google Colab.

## Step 0: Setup — upload the dataset (Colab)

Run the cell below and choose `manufacturing_dataset_1000_samples.csv` from your computer. If you'd rather use Google Drive, mount Drive instead and adjust the path in Step 1.

In [1]:
# Run this cell in Colab to upload the CSV from your computer.
# Skip it if the file is already in your Colab working directory or in Drive.
# IMPORTANT: files.upload() takes NO arguments — it opens a file picker dialog.
try:
    from google.colab import files
    uploaded = files.upload()  # a picker will pop up — select manufacturing_dataset_1000_samples.csv
    print("Uploaded:", list(uploaded.keys()))
except ImportError:
    print("Not running in Colab — make sure the CSV is in the working directory.")

Saving manufacturing_dataset_1000_samples.csv to manufacturing_dataset_1000_samples.csv
Uploaded: ['manufacturing_dataset_1000_samples.csv']


## Step 1: Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
pd.set_option('display.max_columns', None)

In [ ]:
DATA_PATH = 'manufacturing_dataset_1000_samples.csv'  # plain filename — do NOT nest it as 'csv/csv'
import os
assert os.path.isfile(DATA_PATH), f"'{DATA_PATH}' is missing or is a directory. Re-run the upload cell above."
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.info()

## Step 2: Data Exploration

In [ ]:
print("Dataset Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nSummary Statistics:\n")
df.describe(include='all').T

In [ ]:
print("Missing Values per Column:")
missing = df.isnull().sum()
print(missing[missing > 0])

**Business meaning (quick reference):**

| Column | Meaning |
|---|---|
| Timestamp | When the machine cycle was recorded |
| Injection_Temperature | Barrel temperature during injection (°C) |
| Injection_Pressure | Pressure applied during injection (bar) |
| Cycle_Time / Cooling_Time | Time components of one molding cycle (sec) |
| Material_Viscosity | Flow resistance of the molten plastic |
| Ambient_Temperature | Shop-floor temperature |
| Machine_Age | Years the machine has been in service |
| Operator_Experience | Operator experience (months) |
| Maintenance_Hours | Hours since/allocated to maintenance |
| Shift / Machine_Type / Material_Grade / Day_of_Week | Categorical operating context |
| Temperature_Pressure_Ratio, Total_Cycle_Time, Efficiency_Score, Machine_Utilization | Pre-computed derived features |
| **Parts_Per_Hour** | **Target variable** — hourly output |

## Step 3: Handle `Timestamp` and Missing Values

`Timestamp` isn't directly useful for regression, so we extract `Hour` and `DayOfMonth` from it, then drop the raw column. Missing numeric values are filled with the median; missing categorical values (if any) with the mode.

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfMonth'] = df['Timestamp'].dt.day
df = df.drop(columns=['Timestamp'])

numeric_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_all = df.select_dtypes(include=['object']).columns.tolist()

for col in numeric_cols_all:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols_all:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())
print("\nCategorical columns:", categorical_cols_all)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_with_target = df[numeric_cols].corr()['Parts_Per_Hour'].sort_values()

plt.figure(figsize=(8, 6))
corr_with_target.drop('Parts_Per_Hour').plot(kind='barh', color='indigo')
plt.title('Feature Correlation with Parts_Per_Hour')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
target_stats = df['Parts_Per_Hour'].describe()
print("--- Target Variable (Parts_Per_Hour) Summary ---")
print(target_stats)
print(f"Skew: {df['Parts_Per_Hour'].skew():.3f}")

plt.figure(figsize=(8, 5))
sns.histplot(df['Parts_Per_Hour'], kde=True, color='steelblue')
plt.title('Distribution of Parts_Per_Hour')
plt.show()

In [ ]:
df[numeric_cols].hist(bins=30, figsize=(16, 12), edgecolor='black', grid=False)
plt.suptitle("Histograms of Numerical Variables", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
key_params = ['Injection_Temperature', 'Injection_Pressure', 'Cycle_Time',
              'Cooling_Time', 'Material_Viscosity', 'Ambient_Temperature', 'Machine_Utilization']

plt.figure(figsize=(16, 12))
for i, col in enumerate(key_params, 1):
    plt.subplot(3, 3, i)
    plt.scatter(df[col], df['Parts_Per_Hour'], alpha=0.4, s=15)
    plt.xlabel(col)
    plt.ylabel('Parts_Per_Hour')
    plt.title(f'{col} vs Output')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df, x='Machine_Type', y='Parts_Per_Hour', errorbar=None, hue='Machine_Type', palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Average Parts Per Hour by Machine Type")

sns.barplot(data=df, x='Shift', y='Parts_Per_Hour', errorbar=None, hue='Shift', palette="magma", legend=False, ax=axes[1])
axes[1].set_title("Average Parts Per Hour by Shift")
plt.tight_layout()
plt.show()

### Consolidated Dashboard View

One combined figure pulling together the key visuals — useful as a single screenshot for your report/presentation.

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)
fig.suptitle("Manufacturing Dataset — Analysis Dashboard", fontsize=18, fontweight='bold', y=0.98)

# 1. Target distribution
ax1 = fig.add_subplot(gs[0, 0])
sns.histplot(df['Parts_Per_Hour'], kde=True, color='steelblue', ax=ax1)
ax1.set_title("Target Distribution", fontweight='bold')

# 2. Correlation with target (top features)
ax2 = fig.add_subplot(gs[0, 1:])
top_corr = df[numeric_cols].corr()['Parts_Per_Hour'].drop('Parts_Per_Hour').sort_values()
colors = ['crimson' if v < 0 else 'seagreen' for v in top_corr]
top_corr.plot(kind='barh', ax=ax2, color=colors)
ax2.set_title("Feature Correlation with Output", fontweight='bold')
ax2.axvline(0, color='black', linewidth=0.8)

# 3. Correlation heatmap (compact)
ax3 = fig.add_subplot(gs[1, :2])
sns.heatmap(df[numeric_cols].corr(), cmap="coolwarm", center=0, ax=ax3, cbar_kws={'shrink': 0.7},
            xticklabels=True, yticklabels=True, annot=False)
ax3.set_title("Correlation Heatmap", fontweight='bold')
ax3.tick_params(axis='x', rotation=90, labelsize=7)
ax3.tick_params(axis='y', labelsize=7)

# 4. Strongest predictor scatter
ax4 = fig.add_subplot(gs[1, 2])
strongest_feature = top_corr.abs().idxmax()
ax4.scatter(df[strongest_feature], df['Parts_Per_Hour'], alpha=0.4, s=12, color='darkorange')
ax4.set_xlabel(strongest_feature)
ax4.set_ylabel('Parts_Per_Hour')
ax4.set_title(f"Strongest Predictor:\n{strongest_feature}", fontweight='bold', fontsize=10)

# 5. Output by Machine Type
ax5 = fig.add_subplot(gs[2, 0])
sns.barplot(data=df, x='Machine_Type', y='Parts_Per_Hour', errorbar=None, hue='Machine_Type',
            palette="viridis", legend=False, ax=ax5)
ax5.set_title("Output by Machine Type", fontweight='bold')
ax5.tick_params(axis='x', rotation=30)

# 6. Output by Shift
ax6 = fig.add_subplot(gs[2, 1])
sns.barplot(data=df, x='Shift', y='Parts_Per_Hour', errorbar=None, hue='Shift',
            palette="magma", legend=False, ax=ax6)
ax6.set_title("Output by Shift", fontweight='bold')

# 7. Output by Material Grade
ax7 = fig.add_subplot(gs[2, 2])
sns.barplot(data=df, x='Material_Grade', y='Parts_Per_Hour', errorbar=None, hue='Material_Grade',
            palette="crest", legend=False, ax=ax7)
ax7.set_title("Output by Material Grade", fontweight='bold')
ax7.tick_params(axis='x', rotation=30)

plt.savefig('eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved as eda_dashboard.png — download it from the Colab file browser for your report.")

## Step 5: Train-Test Split

In [ ]:
target_var = 'Parts_Per_Hour'
X = df.drop(columns=[target_var])
y = df[target_var]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")
print(f"Numeric features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

## Step 6: Build the Preprocessing + Model Pipeline

One pipeline handles scaling and encoding together, and is reused consistently for training, evaluation, and coefficient interpretation — this is the single source of truth for the rest of the notebook (fixing the original notebook's split-model bug).

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
])

lin_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lin_reg_pipeline.fit(X_train, y_train)
print("Model trained.")

### Optional: compare against Ridge and Lasso (regularized variants)

In [ ]:
models = {
    'Linear Regression': lin_reg_pipeline,
    'Ridge': Pipeline([('preprocessor', preprocessor), ('regressor', Ridge(alpha=1.0))]),
    'Lasso': Pipeline([('preprocessor', preprocessor), ('regressor', Lasso(alpha=0.1))])
}

cv_results = {}
for name, model in models.items():
    if name != 'Linear Regression':
        model.fit(X_train, y_train)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    cv_results[name] = scores
    print(f"{name}: mean CV R² = {scores.mean():.4f} (+/- {scores.std():.4f})")

## Step 7: Model Evaluation

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name="Model"):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    metrics = {}
    for split, y_true, y_pred in [('train', y_train, y_train_pred), ('test', y_test, y_test_pred)]:
        metrics[split] = {
            'R2': r2_score(y_true, y_pred),
            'MSE': mean_squared_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAE': mean_absolute_error(y_true, y_pred)
        }

    print(f"---- {name}: Training Performance ----")
    for k, v in metrics['train'].items():
        print(f"{k}: {v:.4f}")
    print(f"\n---- {name}: Testing Performance ----")
    for k, v in metrics['test'].items():
        print(f"{k}: {v:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    residuals = y_test - y_test_pred
    sns.scatterplot(x=y_test_pred, y=residuals, ax=axes[0])
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel("Predicted Values")
    axes[0].set_ylabel("Residuals")
    axes[0].set_title("Residual Plot (Test Data)")

    sns.scatterplot(x=y_test, y=y_test_pred, ax=axes[1])
    axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
    axes[1].set_xlabel("Actual Values")
    axes[1].set_ylabel("Predicted Values")
    axes[1].set_title("Predicted vs Actual (Test Data)")

    plt.tight_layout()
    plt.show()
    return metrics

metrics = evaluate_model(lin_reg_pipeline, X_train, y_train, X_test, y_test, name="Linear Regression")

## Step 8: Feature Importance & Business Interpretation

Coefficients come from the *same* fitted pipeline used above, so they match the evaluated model exactly.

In [ ]:
feature_names = lin_reg_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefficients = lin_reg_pipeline.named_steps['regressor'].coef_
clean_feature_names = [f.split('__', 1)[-1] for f in feature_names]

coef_df = pd.DataFrame({
    'Feature': clean_feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
}).sort_values(by='Abs_Coefficient', ascending=False).reset_index(drop=True)

print("--- Feature Coefficients Ranked by Impact ---")
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

plt.figure(figsize=(9, 8))
sns.barplot(data=coef_df.head(15), x='Coefficient', y='Feature', hue='Feature', palette='crest', legend=False)
plt.title('Top 15 Features by Coefficient Magnitude')
plt.tight_layout()
plt.show()

In [ ]:
print("BUSINESS IMPLICATIONS (top 5 features):\n")
for _, row in coef_df.head(5).iterrows():
    feature, coef = row['Feature'], row['Coefficient']
    direction = "increases" if coef > 0 else "decreases"
    print(f"- {feature} ({coef:+.4f}): a one-unit (scaled) increase {direction} predicted output by {abs(coef):.3f} parts/hour, holding other factors constant.")

## Step 9: Optimal Operating Ranges (IQR-based)

Uses the 25th–75th percentile of each key parameter, computed on the **training data only** to avoid leaking test-set information.

In [ ]:
key_features = ['Injection_Temperature', 'Injection_Pressure', 'Cycle_Time', 'Cooling_Time', 'Material_Viscosity']
iqr_bounds = {}
for col in key_features:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr_bounds[col] = (q1, q3)

print("=== Suggested Optimal Operating Ranges (25th-75th percentile, training data) ===")
for col, (low, high) in iqr_bounds.items():
    print(f"{col:22s}: {low:.2f} to {high:.2f}")

In [ ]:
def suggest_setting_adjustments(sample_row, bounds):
    """Compares a machine's current settings against IQR bounds and returns adjustment suggestions."""
    adjustments = []
    for param, (low, high) in bounds.items():
        val = sample_row[param]
        if val < low:
            adjustments.append(f"Increase {param} from {val:.1f} (target: {low:.1f}-{high:.1f})")
        elif val > high:
            adjustments.append(f"Decrease {param} from {val:.1f} (target: {low:.1f}-{high:.1f})")
    return adjustments if adjustments else ["All monitored parameters are within the optimal range."]

sample = X_test.iloc[0]
print(f"Example — Test sample #0 adjustment suggestions:")
for line in suggest_setting_adjustments(sample, iqr_bounds):
    print(" -", line)

## Step 10: Recommendations Summary

- **Focus optimization on the top-ranked features** from Step 8 — those give the fastest gains in `Parts_Per_Hour`.
- **Keep process parameters within the IQR bands** from Step 9 to sustain output without introducing defects.
- **Flag out-of-band readings in real time** so underperforming machines/shifts are caught early.
- **Preventive maintenance and operator training** should target machines/operators repeatedly drifting outside the optimal ranges.

## Step 11: Save the Trained Model

In [ ]:
import joblib
joblib.dump(lin_reg_pipeline, 'manufacturing_output_model.pkl')
print("Model saved to manufacturing_output_model.pkl")

# In Colab, download it to your machine:
try:
    from google.colab import files
    files.download('manufacturing_output_model.pkl')
except ImportError:
    pass

## Step 12 (Optional): Deploy as a Streamlit App

This writes a Streamlit app (`app.py`) with **two tabs**:
- **Data Visualization** — distributions, correlation heatmap, feature-vs-output scatter plots, categorical comparisons (this is what shows up on the web page as graphs).
- **Predict Output** — enter process parameters and get a live prediction.

Run the cells below **in Colab, after Step 1–3 above have created `manufacturing_dataset_1000_samples.csv` in the working directory**.

In [ ]:
!pip install -q streamlit plotly

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

st.set_page_config(page_title="Manufacturing Yield Dashboard", layout="wide")

@st.cache_data
def load_and_prep_data():
    df = pd.read_csv('manufacturing_dataset_1000_samples.csv')
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df['Hour'] = df['Timestamp'].dt.hour
    df['DayOfMonth'] = df['Timestamp'].dt.day
    df = df.drop(columns=['Timestamp'])
    df = df.fillna(df.median(numeric_only=True))
    return df

@st.cache_resource
def train_model(df):
    target_var = 'Parts_Per_Hour'
    X = df.drop(columns=[target_var])
    y = df[target_var]
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])
    pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
    pipeline.fit(X, y)
    return pipeline

df = load_and_prep_data()
model = train_model(df)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

st.title("Manufacturing Yield Dashboard")

tab_viz, tab_predict = st.tabs(["Data Visualization", "Predict Output"])

# ---------------- TAB 1: DATA VISUALIZATION ----------------
with tab_viz:
    st.subheader("Dataset Overview")
    c1, c2, c3 = st.columns(3)
    c1.metric("Rows", df.shape[0])
    c2.metric("Avg Parts/Hour", f"{df['Parts_Per_Hour'].mean():.1f}")
    c3.metric("Machine Types", df['Machine_Type'].nunique())

    st.markdown("### Target Distribution")
    fig_hist = px.histogram(df, x="Parts_Per_Hour", nbins=40, marginal="box",
                             title="Distribution of Parts_Per_Hour")
    st.plotly_chart(fig_hist, use_container_width=True)

    st.markdown("### Correlation with Target")
    corr = df[numeric_cols].corr()["Parts_Per_Hour"].drop("Parts_Per_Hour").sort_values()
    fig_corr_bar = px.bar(corr, orientation="h", title="Feature Correlation with Parts_Per_Hour",
                           labels={"value": "Correlation", "index": "Feature"})
    st.plotly_chart(fig_corr_bar, use_container_width=True)

    st.markdown("### Full Correlation Heatmap")
    fig_heatmap = px.imshow(df[numeric_cols].corr(), text_auto=".2f", color_continuous_scale="RdBu_r",
                             aspect="auto", title="Correlation Heatmap")
    st.plotly_chart(fig_heatmap, use_container_width=True)

    st.markdown("### Feature vs. Output")
    scatter_feature = st.selectbox("Choose a feature to plot against Parts_Per_Hour",
                                    [c for c in numeric_cols if c != "Parts_Per_Hour"])
    fig_scatter = px.scatter(df, x=scatter_feature, y="Parts_Per_Hour", trendline="ols",
                              opacity=0.5, title=f"{scatter_feature} vs. Parts_Per_Hour")
    st.plotly_chart(fig_scatter, use_container_width=True)

    st.markdown("### Output by Category")
    cat_choice = st.selectbox("Choose a categorical column", categorical_cols)
    fig_bar = px.bar(df.groupby(cat_choice)["Parts_Per_Hour"].mean().reset_index(),
                      x=cat_choice, y="Parts_Per_Hour",
                      title=f"Average Parts_Per_Hour by {cat_choice}")
    st.plotly_chart(fig_bar, use_container_width=True)

    with st.expander("View raw data sample"):
        st.dataframe(df.head(50))

# ---------------- TAB 2: PREDICTION ----------------
with tab_predict:
    st.subheader("Predict Machine Output")
    st.write("Enter process parameters to predict expected Parts Per Hour.")

    with st.form("prediction_form"):
        col1, col2 = st.columns(2)
        with col1:
            input_temp = st.number_input("Injection Temperature (C)", 100.0, 350.0, 215.0)
            input_pressure = st.number_input("Injection Pressure (bar)", 50.0, 250.0, 116.0)
            input_cycle = st.number_input("Cycle Time (sec)", 5.0, 120.0, 35.0)
        with col2:
            input_cooling = st.number_input("Cooling Time (sec)", 1.0, 60.0, 12.0)
            input_exp_years = st.number_input("Operator Experience (years)", 0.1, 20.0, 5.0)
            input_machine_type = st.selectbox("Machine Type", sorted(df['Machine_Type'].unique()))
        submit = st.form_submit_button("Calculate Output")

    if submit:
        row = {col: df[col].median() if df[col].dtype != object else df[col].mode()[0]
               for col in df.drop(columns=['Parts_Per_Hour']).columns}
        row.update({
            'Injection_Temperature': input_temp,
            'Injection_Pressure': input_pressure,
            'Cycle_Time': input_cycle,
            'Cooling_Time': input_cooling,
            'Operator_Experience': input_exp_years * 12,
            'Machine_Type': input_machine_type,
            'Temperature_Pressure_Ratio': input_temp / input_pressure,
            'Total_Cycle_Time': input_cycle + input_cooling,
        })
        single_input = pd.DataFrame([row])
        prediction = model.predict(single_input)[0]
        st.success(f"Predicted Parts_Per_Hour: {prediction:.1f}")

### Launching the app in Colab

**Recommended: Colab's built-in port proxy** — no external tunneling service, so no signup and no third-party 502/WebSocket issues:

In [ ]:
!pkill -f streamlit
import time
time.sleep(2)
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --server.headless true \
  &>/content/logs.txt &
time.sleep(6)
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

Click the printed link. If the page is blank or shows connection errors, check what Streamlit itself logged:
```python
!cat /content/logs.txt
```

**Fallback — localtunnel** (if the Colab proxy link doesn't work in your network/browser):
```python
!pkill -f streamlit
!streamlit run app.py --server.enableWebsocketCompression false &>/content/logs.txt &
!npx localtunnel --port 8501
```
Open the printed `https://*.loca.lt` link, and click through the "Click to Continue" interstitial if it appears.